# Try the personal-card-agent agent

Two ways to try this repo's agent from a notebook:

- **Option A - quick local invoke.** Builds the agent in-process with the
  OSS `deepagents` library, reusing this repo's `instructions.md` and
  `middleware/` (customer-context seeding + tool-call logging). No `mda`
  CLI or server needed - fastest way to iterate.
- **Option B - talk to the real managed agent.** Connects to the agent
  exactly as `agent.py` defines it, served locally by `mda dev .`.

**Prerequisites**

```bash
uv sync --extra notebook
# fill OPENAI_API_KEY (and LANGSMITH_API_KEY for option B) in ../.env
jupyter notebook notebooks/try_agent.ipynb
```

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

# Notebook runs from notebooks/, but the agent's code (middleware/, data/,
# instructions.md, .env) lives one level up at the repo root.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

import os

assert os.environ.get("OPENAI_API_KEY"), (
    f"Set OPENAI_API_KEY in {REPO_ROOT / '.env'} first"
)

## Option A - quick local invoke (no `mda dev` needed)

In [ ]:
from deepagents import create_deep_agent

from middleware.audit import log_tool_calls
from middleware.seed_customer_context import SeedCustomerContext

instructions = (REPO_ROOT / "instructions.md").read_text()

local_agent = create_deep_agent(
    model="openai:gpt-5.5",
    tools=[],
    middleware=[log_tool_calls, SeedCustomerContext()],
    system_prompt=instructions,
)

result = local_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Tháng này tôi đã chi tiêu bao nhiêu trên thẻ?"}
        ]
    }
)
print(result["messages"][-1].content)

## Option B - talk to the managed agent via `mda dev`

In a separate terminal, from the repo root, run:

```bash
mda dev .
```

It prints the local Agent Server URL (default `http://127.0.0.1:2024`) and
the graph/assistant name to use below - it's usually the agent `name` from
`agent.py` (`"personal-card-agent"`), but check the printed Studio URL /
server logs and adjust `ASSISTANT_ID` if it differs.

In [ ]:
from langgraph_sdk import get_sync_client

SERVER_URL = "http://127.0.0.1:2024"
ASSISTANT_ID = "personal-card-agent"

client = get_sync_client(url=SERVER_URL)

thread = client.threads.create()
for chunk in client.runs.stream(
    thread["thread_id"],
    ASSISTANT_ID,
    input={
        "messages": [
            {"role": "user", "content": "Tháng này tôi đã chi tiêu bao nhiêu trên thẻ?"}
        ]
    },
    stream_mode="values",
):
    if chunk.event == "values":
        print(chunk.data["messages"][-1]["content"])